# So sánh kNN, WKNN (1/d) và WKNN (1/d²) trên UJIIndoorLoc  — Lần 4
**Mục tiêu:** xem khi thay bước gộp tọa độ của kNN (trọng số đều `1/k`) bằng hai dạng WKNN thì **success rate, sai số vị trí và thời gian** thay đổi thế nào, và **tăng bậc của khoảng cách từ 1 lên 2 có khác gì không**.

| Ký hiệu trong code | Tên trong báo cáo | Trọng số của láng giềng thứ i (đã chuẩn hóa Σw = 1) |
|---|---|---|
| `knn` | kNN | `1/k` (trọng số đều) |
| `idw` | WKNN (1/d) — *WKNN cơ bản, chính là "IDW-WKNN" ở Lần 3* | `(1/dᵢ) / Σ(1/dⱼ)` |
| `idw2` | WKNN (1/d²) — *theo hướng của Xiang et al. 2023* | `(1/dᵢ²) / Σ(1/dⱼ²)` |

Ba phương pháp thực chất là một dãy theo bậc `p` của trọng số `1/dᵖ`: **p = 0 (kNN) → p = 1 → p = 2**.

> **Lưu ý tên gọi:** trong bài Xiang et al. 2023, "IDW" chủ yếu là phép **nội suy để tạo thêm fingerprint** (có dùng bình phương), còn WKNN ở bước định vị dùng `1/(D+ε)`. Notebook này **không** nội suy CSDL và **không** dùng GM(1,1); chỉ so sánh cách đặt trọng số khi gộp tọa độ. Vì vậy ở đây dùng tên "WKNN (1/d)" và "WKNN (1/d²)" để tránh nhầm với nội suy IDW.

**Giữ nguyên ở cả ba phương pháp:** cùng dataset UJIIndoorLoc (train → tìm láng giềng, validation → kiểm tra), biểu diễn positive → powed, khoảng cách Sørensen, vote tòa → tầng, quét **k = 1 → 20**. Chỉ khác đúng một chỗ: cách tính trọng số khi gộp tọa độ. Nếu có láng giềng d ≈ 0 thì lấy trung bình các láng giềng đó (tránh chia cho 0).

**Cấu trúc notebook**
| Phần | Nội dung |
|---|---|
| Bước 0–4 | Tải dữ liệu, tiền xử lý, tìm láng giềng, hàm dự đoán, chạy quét k |
| **Phần A** | Bảng giải thích các chỉ số (đọc trước khi xem kết quả) |
| **Phần B** | Kết quả dạng bảng |
| **Phần C** | Biểu đồ (mỗi biểu đồ một ô riêng, có chú thích cách đọc) |
| **Phần D** | Đánh giá: ba cặp so sánh kNN–WKNN(1/d), kNN–WKNN(1/d²), WKNN(1/d)–WKNN(1/d²) |
| **Phần E** | Lưu file kết quả |

⚠️ **Hạn chế cần ghi trong báo cáo:** UJIIndoorLoc chỉ có tập train và validation, nên k tốt nhất được chọn ngay trên tập kiểm tra (giống Lần 2, 3). Kết quả hơi lạc quan. Vì vậy Phần D còn so sánh các phương pháp ở **cùng k**, không phụ thuộc cách chọn k.

In [ ]:
!pip install -q indoorloc
from indoorloc.datasets import UJIndoorLocDataset
import numpy as np, pandas as pd, time

train_ds = UJIndoorLocDataset(split="train", download=True, normalize=False)
val_ds = UJIndoorLocDataset(split="test", download=True, normalize=False)

def dataset_to_df(ds):
    X = np.array([s.to_numpy() for s in ds.signals])
    df = pd.DataFrame(X, columns=[f"WAP{i+1:03d}" for i in range(X.shape[1])])
    df["LONGITUDE"] = [loc.coordinate.x for loc in ds.locations]
    df["LATITUDE"] = [loc.coordinate.y for loc in ds.locations]
    df["FLOOR"] = [int(loc.floor) for loc in ds.locations]
    df["BUILDINGID"] = [int(loc.building_id) for loc in ds.locations]
    return df

train_df, val_df = dataset_to_df(train_ds), dataset_to_df(val_ds)
wap_cols = [c for c in train_df.columns if c.startswith("WAP")]
print("Training:", train_df.shape, "| Validation (test):", val_df.shape)

## Bước 1 — Biểu diễn dữ liệu (giữ nguyên code cũ)

In [ ]:
NOT_DETECTED, MIN_RSS = 100, -104
def to_positive(X, min_rss=MIN_RSS):
    out = X.copy().astype(float); det = X != NOT_DETECTED; m = min_rss - 1
    out[det] = X[det] - m; out[~det] = 0.0
    return out
def to_powed(X, min_rss=MIN_RSS, beta=np.e):
    return (to_positive(X, min_rss) ** beta) / ((-(min_rss - 1)) ** beta)

X_train = to_powed(train_df[wap_cols].to_numpy()); X_val = to_powed(val_df[wap_cols].to_numpy())
tb, tf = train_df.BUILDINGID.to_numpy(int), train_df.FLOOR.to_numpy(int)
tx, ty = train_df.LONGITUDE.to_numpy(), train_df.LATITUDE.to_numpy()
vb, vf = val_df.BUILDINGID.to_numpy(int), val_df.FLOOR.to_numpy(int)
vx, vy = val_df.LONGITUDE.to_numpy(), val_df.LATITUDE.to_numpy()
NB, NF = int(max(tb.max(), vb.max())) + 1, int(max(tf.max(), vf.max())) + 1
REF = (tb, tf, tx, ty)   # du lieu tham chieu (train)
raw_tr, raw_va = train_df[wap_cols].to_numpy(), val_df[wap_cols].to_numpy()
print("RSSI da phat hien nho nhat (train/val):", raw_tr[raw_tr != 100].min(), raw_va[raw_va != 100].min(), "(MIN_RSS =", MIN_RSS, ")")

## Bước 2 — Tìm k láng giềng gần nhất (Sørensen), một lần với k = 20 rồi cắt cho từng k
Bước này **giống hệt nhau** ở cả ba phương pháp nên chỉ đo thời gian một lần (`T_SEARCH`) và cộng vào cả ba.

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
K_MAX = 20

def knn_search(Q, R, K, chunk=500):
    n = len(Q); idx = np.empty((n, K), np.int32); dist = np.empty((n, K))
    for s in range(0, n, chunk):
        D = np.nan_to_num(pairwise_distances(Q[s:s+chunk], R, metric="braycurtis", n_jobs=-1), nan=0.0)  # Sorensen
        o = np.argsort(D, axis=1, kind="stable")[:, :K]
        idx[s:s+chunk] = o; dist[s:s+chunk] = np.take_along_axis(D, o, 1)
    return dist, idx

# kiem tra tuong duong voi ham sorensen cu
def sorensen_old(u, v): return np.sum(np.abs(u - v)) / (np.sum(u + v) + 1e-12)
d_new, _ = knn_search(X_val[:10], X_train[:3000], 5)
d_old, _ = NearestNeighbors(n_neighbors=5, metric=sorensen_old, algorithm="brute").fit(X_train[:3000]).kneighbors(X_val[:10])
print("Khoang cach moi == ham Sorensen cu:", np.allclose(d_new, d_old, atol=1e-9))

t0 = time.perf_counter()
distances_all, indices_all = knn_search(X_val, X_train, K_MAX)
T_SEARCH = time.perf_counter() - t0
print(f"Da truy van {len(X_val)} mau, k_max={K_MAX}: {T_SEARCH:.1f}s (T_SEARCH, dung chung cho ca hai phuong phap)")

## Bước 3 — Dự đoán: kNN vs WKNN (1/d) vs WKNN (1/d²) (chỉ khác cách tính trọng số)
Tòa/tầng: vote không trọng số, giống nhau ở cả ba. Tọa độ: trung bình các láng giềng cùng tòa+tầng với trọng số
`wᵢ = (1/dᵢᵖ) / Σ(1/dⱼᵖ)`, trong đó **p = 0** (kNN), **p = 1** (WKNN 1/d), **p = 2** (WKNN 1/d²).

Ví dụ d = (2, 4, 8): kNN → 33% / 33% / 33%; p = 1 → 57% / 29% / 14%; p = 2 → 76% / 19% / 5%.
Bậc càng cao thì láng giềng gần càng được ưu tiên (p → ∞ tiến về k = 1).

In [ ]:
TOL = 1e-12
METHODS = ("knn", "idw", "idw2")
NAME = {"knn": "kNN", "idw": "WKNN (1/d)", "idw2": "WKNN (1/d²)"}
POWER = {"idw": 1, "idw2": 2}          # bậc p của trọng số 1/d^p (kNN không dùng)

def predict(indices, dists, k, method, ref=REF):
    rb, rf, rx, ry = ref
    nb, d = indices[:, :k], dists[:, :k]
    b, f = rb[nb], rf[nb]
    pb = np.stack([(b == c).sum(1) for c in range(NB)], 1).argmax(1)
    mb = b == pb[:, None]
    pf = np.stack([((f == c) & mb).sum(1) for c in range(NF)], 1).argmax(1)
    m = mb & (f == pf[:, None])
    if method == "knn":
        w, n0 = m / m.sum(1, keepdims=True), 0
    else:
        p = POWER[method]
        z = m & (d <= TOL); has0 = z.any(1)                      # d = 0 -> trung binh cac lang gieng d ~ 0
        inv = np.where(m & ~z, 1.0 / np.maximum(d, TOL) ** p, 0.0)
        w = np.where(has0[:, None], z / np.maximum(z.sum(1, keepdims=True), 1),
                     inv / np.maximum(inv.sum(1, keepdims=True), 1e-300))
        n0 = int(has0.sum())
    return pb, pf, (w * rx[nb]).sum(1), (w * ry[nb]).sum(1), n0

def evaluate(pb, pf, px, py):
    ok = (pb == vb) & (pf == vf); e_all = np.hypot(px - vx, py - vy); e = e_all[ok]
    return dict(success_rate=ok.mean()*100, mean_error=e.mean(), median=np.median(e), rmse=np.sqrt((e**2).mean()),
                p90=np.percentile(e, 90), le1=(e<=1).mean()*100, le3=(e<=3).mean()*100, le5=(e<=5).mean()*100), ok, e_all

# --- kiem tra tay: d=(2,4,8), P1=(0,0) P2=(10,0) P3=(0,10) ---
#   p=1: w=(4/7,2/7,1/7)    -> x=20/7, y=10/7
#   p=2: w=(16/21,4/21,1/21) -> x=40/21, y=10/21
ref3 = (np.zeros(3, int), np.zeros(3, int), np.array([0., 10, 0]), np.array([0., 0, 10]))
i3, d3 = np.array([[0, 1, 2]]), np.array([[2., 4, 8]])
xy = lambda r: (float(r[2][0]), float(r[3][0]))            # lay (x, y) cua mau dau tien
w1 = (1/d3[0]) / (1/d3[0]).sum(); w2 = (1/d3[0]**2) / (1/d3[0]**2).sum()
assert np.isclose(w1.sum(), 1) and np.all(np.diff(w1) < 0)
assert np.isclose(w2.sum(), 1) and np.all(np.diff(w2) < 0) and w2[0] > w1[0]      # p=2 uu tien lang gieng gan hon
assert np.allclose(xy(predict(i3, d3, 3, "idw", ref3)), (20/7, 10/7))
assert np.allclose(xy(predict(i3, d3, 3, "idw2", ref3)), (40/21, 10/21))
assert np.allclose(xy(predict(i3, d3, 3, "knn", ref3)), (10/3, 10/3))
for mm in ("idw", "idw2"):                                                         # k=1: moi phuong phap == kNN
    assert np.allclose(xy(predict(i3, d3, 1, mm, ref3)), xy(predict(i3, d3, 1, "knn", ref3)))
    r0 = predict(i3, np.array([[0., 4, 8]]), 3, mm, ref3); assert np.allclose(xy(r0), (0, 0)) and r0[4] == 1   # d=0
print("Kiem tra tay OK: tong w = 1, w giam khi d tang, p=2 > p=1 o lang gieng gan nhat, k=1 == kNN, d=0 khong chia cho 0")

## Bước 4 — Quét k = 1 → 20 cho cả ba phương pháp (kèm thời gian)
Thời gian gộp tọa độ đo bằng `min` của 3 lần chạy để giảm nhiễu. `total_s = T_SEARCH + t_predict`. Tiêu chí chọn k tốt nhất như Lần 2, 3: success rate cao nhất, hòa thì sai số thấp hơn.

In [ ]:
rows, R = [], {}
for method in METHODS:
    for k in range(1, K_MAX + 1):
        ts = []
        for _ in range(3):
            t0 = time.perf_counter(); out = predict(indices_all, distances_all, k, method); ts.append(time.perf_counter() - t0)
        pb, pf, px, py, n0 = out
        r, ok, e_all = evaluate(pb, pf, px, py)
        t_pred = min(ts)
        rows.append(dict(method=method, k=k, n_query_d0=n0, **r, t_predict_s=t_pred,
                         total_s=T_SEARCH + t_pred, us_per_sample=(T_SEARCH + t_pred) / len(vx) * 1e6))
        R[(method, k)] = (ok, e_all)
sweep = pd.DataFrame(rows)
BEST = {m: int(sweep[sweep.method == m].sort_values(["success_rate", "mean_error"], ascending=[False, True]).iloc[0].k) for m in METHODS}
print(f"Đã chạy xong: {len(METHODS)} phương pháp × {K_MAX} giá trị k = {len(sweep)} lần đánh giá trên {len(vx)} mẫu kiểm tra.")
print("k tốt nhất (success rate cao nhất, hòa thì sai số thấp hơn): " + " | ".join(f"{NAME[m]} k={BEST[m]}" for m in METHODS))

# PHẦN A — Các chỉ số trong kết quả nghĩa là gì?
Mỗi mẫu kiểm tra là một lần đo Wi-Fi tại một vị trí đã biết thật. Mô hình đoán **tòa nhà, tầng và tọa độ (x, y)**, rồi so với vị trí thật.

| Chỉ số | Ý nghĩa | Tốt là |
|---|---|---|
| **k** | Số láng giềng (fingerprint giống nhất trong tập huấn luyện) dùng để đoán vị trí. | — (tham số, được quét từ 1 đến 20) |
| **p (bậc trọng số)** | Trọng số láng giềng ∝ 1/dᵖ. p = 0: kNN; p = 1: WKNN (1/d); p = 2: WKNN (1/d²). p càng lớn, láng giềng gần càng chiếm ưu thế. | — |
| **Success rate (%)** | Tỉ lệ mẫu mà mô hình đoán **đúng cả tòa nhà lẫn tầng**. Ví dụ 95% nghĩa là 95/100 mẫu được xác định đúng tòa và tầng. | Cao |
| **Mean error (m)** | **Sai số vị trí trung bình**: khoảng cách thẳng (mét) giữa vị trí đoán và vị trí thật, lấy trung bình. *Chỉ tính trên các mẫu đã đúng tòa và tầng* (theo giao thức của Torres-Sospedra et al., 2015). | Thấp |
| **Median (m)** | Sai số **trung vị**: một nửa số mẫu có sai số nhỏ hơn giá trị này. Ít bị ảnh hưởng bởi vài mẫu sai rất lớn. | Thấp |
| **RMSE (m)** | Căn bậc hai của trung bình bình phương sai số. Phạt nặng các mẫu sai lớn hơn mean error. | Thấp |
| **P90 (m)** | Phân vị 90: 90% số mẫu có sai số nhỏ hơn giá trị này. Cho biết độ tin cậy ở trường hợp xấu. | Thấp |
| **≤1 m, ≤3 m, ≤5 m (%)** | Tỉ lệ mẫu có sai số không vượt quá 1, 3, 5 mét. | Cao |
| **CDF sai số** | Đường cong cho biết "bao nhiêu % mẫu có sai số ≤ x mét". Đường nằm **càng cao và càng lệch trái** thì mô hình càng chính xác. | — |
| **Thời gian (s)** | `t_predict_s`: thời gian bước gộp tọa độ (chỗ duy nhất các phương pháp khác nhau). `total_s` = thời gian tìm láng giềng (dùng chung, đo một lần) + `t_predict_s`. `us_per_sample`: micro-giây trung bình cho mỗi mẫu. | Thấp |
| **n_query_d0** | Số mẫu có láng giềng trùng khít (khoảng cách 0), WKNN xử lý riêng để không chia cho 0. Với kNN luôn bằng 0. | — |
| **d_mean (A − B)** | Hiệu sai số trung bình giữa phương pháp A và B ở cùng k. **Dương = B chính xác hơn A**, âm = B kém hơn A. Ba cặp: kNN − WKNN(1/d), kNN − WKNN(1/d²), WKNN(1/d) − WKNN(1/d²). | Dương |
| **Wilcoxon p-value** | Kiểm định thống kê xem chênh lệch giữa hai phương pháp có thật hay chỉ do ngẫu nhiên. Vì so 3 cặp nên dùng ngưỡng hiệu chỉnh **p < 0,05/3 ≈ 0,0167** (Bonferroni). | — |

**Lưu ý quan trọng:** vote tòa/tầng giống hệt nhau ở cả ba phương pháp nên **success rate luôn bằng nhau ở cùng k**. WKNN chỉ có thể làm thay đổi **sai số tọa độ** (và thời gian).

# PHẦN B — Kết quả dạng bảng

### B1. Success rate theo k
Tỉ lệ (%) đoán đúng tòa + tầng. Các cột giống nhau là bình thường (xem lưu ý ở Phần A).

In [ ]:
t_sr = sweep.pivot(index="k", columns="method", values="success_rate").rename(columns=NAME)[[NAME[m] for m in METHODS]]
t_sr.columns = [f"Success rate (%) - {c}" for c in t_sr.columns]
display(t_sr.round(2))

### B2. Sai số vị trí theo k (mét)
Mean error, Median và P90 của ba phương pháp. Chỉ tính trên các mẫu đúng tòa + tầng. Số nhỏ hơn là tốt hơn.

In [ ]:
t_err = sweep.pivot(index="k", columns="method", values=["mean_error", "median", "p90"])
t_err = t_err.rename(columns={"mean_error": "Mean error (m)", "median": "Median (m)", "p90": "P90 (m)", **NAME})
t_err = t_err.reindex(columns=pd.MultiIndex.from_product([["Mean error (m)", "Median (m)", "P90 (m)"], [NAME[m] for m in METHODS]]))
display(t_err.round(3))

### B3. Thời gian theo k (giây, cho toàn bộ tập kiểm tra)
`Gộp tọa độ` là phần duy nhất các phương pháp khác nhau; `Tổng` gồm cả bước tìm láng giềng dùng chung.

In [ ]:
t_time = sweep.pivot(index="k", columns="method", values=["t_predict_s", "total_s"])
t_time = t_time.rename(columns={"t_predict_s": "Gộp tọa độ (s)", "total_s": "Tổng (s)", **NAME})
t_time = t_time.reindex(columns=pd.MultiIndex.from_product([["Gộp tọa độ (s)", "Tổng (s)"], [NAME[m] for m in METHODS]]))
display(t_time.round(4))
print(f"Thời gian tìm láng giềng (dùng chung cho cả ba, đo một lần): {T_SEARCH:.2f} s cho {len(vx)} mẫu")

### B4. Bảng tổng hợp tại các k quan trọng
Gồm: k tốt nhất của từng phương pháp, kNN và WKNN (1/d) ở đúng k tốt nhất của WKNN (1/d²) (để so công bằng), và k = 13 (giá trị tham chiếu từ tài liệu Torres-Sospedra et al., 2015).

In [ ]:
cols = ["success_rate", "mean_error", "median", "rmse", "p90", "le1", "le3", "le5", "t_predict_s", "total_s", "us_per_sample", "n_query_d0"]
plan = [("knn", BEST["knn"], "kNN, k tốt nhất"), ("idw", BEST["idw"], "WKNN (1/d), k tốt nhất"), ("idw2", BEST["idw2"], "WKNN (1/d²), k tốt nhất"),
        ("knn", BEST["idw2"], "kNN, cùng k với WKNN (1/d²)"), ("idw", BEST["idw2"], "WKNN (1/d), cùng k với WKNN (1/d²)"),
        ("knn", 13, "kNN, k=13 (tham chiếu)"), ("idw", 13, "WKNN (1/d), k=13 (tham chiếu)"), ("idw2", 13, "WKNN (1/d²), k=13 (tham chiếu)")]
seen, out = set(), []
for m, k, tag in plan:
    if (m, k) in seen: continue
    seen.add((m, k)); out.append({"Phương pháp": NAME[m], "k": k, "Ghi chú": tag,
        **sweep[(sweep.method == m) & (sweep.k == k)].iloc[0][cols].to_dict()})
table = pd.DataFrame(out).rename(columns={"success_rate": "Success rate (%)", "mean_error": "Mean (m)", "median": "Median (m)", "rmse": "RMSE (m)",
    "p90": "P90 (m)", "le1": "≤1m (%)", "le3": "≤3m (%)", "le5": "≤5m (%)", "t_predict_s": "Gộp tọa độ (s)", "total_s": "Tổng (s)",
    "us_per_sample": "µs/mẫu", "n_query_d0": "Số mẫu d=0"})
display(table.round(3))

# PHẦN C — Biểu đồ
Mỗi biểu đồ nằm trong một ô riêng, có chú thích cách đọc phía trên. Màu **cam = kNN**, **xanh dương = WKNN (1/d)**, **xanh lá = WKNN (1/d²)**.

In [ ]:
import matplotlib.pyplot as plt
COL = {"knn": "#DD8452", "idw": "#4C72B0", "idw2": "#55A868"}
def line_chart(col, title, ylabel, fname, mark_best=False):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for m in METHODS:
        s = sweep[sweep.method == m]
        ax.plot(s.k, s[col], "o-", color=COL[m], label=NAME[m], ms=4)
        if mark_best: ax.axvline(BEST[m], color=COL[m], ls="--", alpha=.5, label=f"k tốt nhất {NAME[m]} = {BEST[m]}")
    ax.set_title(title); ax.set_xlabel("k (số láng giềng)"); ax.set_ylabel(ylabel); ax.set_xticks(range(1, K_MAX + 1)); ax.grid(alpha=.3); ax.legend()
    fig.tight_layout(); fig.savefig(fname, dpi=150); plt.show()
print("Sẵn sàng vẽ biểu đồ.")

### C1. Success rate theo k
**Cách đọc:** trục ngang là k, trục dọc là % mẫu đúng tòa + tầng. Ba đường **trùng nhau** vì vote tòa/tầng giống hệt nhau (không phải lỗi). Dùng biểu đồ này để thấy k nào cho success rate cao nhất.

In [ ]:
line_chart("success_rate", "Success rate (đúng tòa + tầng) theo k", "Success rate (%)", "L4_C1_success_rate.png")

### C2. Sai số vị trí trung bình theo k
**Cách đọc:** đường **thấp hơn = chính xác hơn**. Đường đứt nét đánh dấu k tốt nhất của từng phương pháp. Ở k = 1 ba đường trùng nhau (chỉ có một láng giềng nên không có trọng số). Khi k tăng, WKNN (1/d²) thường nằm gần k nhỏ hơn vì ưu tiên láng giềng gần mạnh hơn.

In [ ]:
line_chart("mean_error", "Sai số vị trí trung bình theo k", "Mean error (m)", "L4_C2_mean_error.png", mark_best=True)

### C3. Sai số trung vị (median) theo k
**Cách đọc:** như C2 nhưng dùng trung vị, ít bị ảnh hưởng bởi vài mẫu sai rất lớn. Nếu C2 và C3 cùng cho một kết luận thì kết luận đáng tin hơn.

In [ ]:
line_chart("median", "Sai số trung vị theo k", "Median error (m)", "L4_C3_median.png")

### C4. Phân bố sai số (CDF) tại k tốt nhất của mỗi phương pháp
**Cách đọc:** tại một điểm x trên trục ngang, chiều cao của đường là tỉ lệ mẫu có sai số ≤ x mét. **Đường nằm cao hơn / lệch trái hơn = tốt hơn** ở mức sai số đó. Các đường sát nhau nghĩa là các phương pháp gần như tương đương.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for m in METHODS:
    ok, e_all = R[(m, BEST[m])]; e = np.sort(e_all[ok])
    ax.plot(e, np.arange(1, len(e) + 1) / len(e), color=COL[m], label=f"{NAME[m]} (k={BEST[m]})")
ax.set_xlim(0, 40); ax.set_xlabel("Sai số vị trí (m)"); ax.set_ylabel("Tỉ lệ mẫu có sai số ≤ x"); ax.set_title("CDF sai số (chỉ tính mẫu đúng tòa + tầng)")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout(); fig.savefig("L4_C4_cdf.png", dpi=150); plt.show()

### C5. Thời gian chạy theo k
**Cách đọc:** đường thấp hơn = nhanh hơn. Thời gian tổng gồm bước tìm láng giềng (chiếm gần hết, giống nhau ở cả ba phương pháp) nên các đường thường rất sát nhau.

In [ ]:
line_chart("total_s", "Tổng thời gian chạy theo k (toàn bộ tập kiểm tra)", "Thời gian (s)", "L4_C5_time.png")

### C6. Chênh lệch sai số giữa các cặp phương pháp ở từng k
**Cách đọc:** mỗi cột là `sai số phương pháp A − sai số phương pháp B` ở một giá trị k. **Cột dương = B chính xác hơn A** ở k đó, cột âm = B kém hơn. Cặp quan trọng nhất cho câu hỏi "tăng bậc từ 1 lên 2 có khác gì không" là **WKNN (1/d) − WKNN (1/d²)**. Nếu các cột này rất thấp (gần 0) thì hai dạng trọng số gần như tương đương.

In [ ]:
PAIRS = [("knn", "idw"), ("knn", "idw2"), ("idw", "idw2")]
PCOL = {("knn", "idw"): "#4C72B0", ("knn", "idw2"): "#55A868", ("idw", "idw2"): "#8172B3"}
piv = sweep.pivot(index="k", columns="method", values=["mean_error", "median", "p90", "total_s"])
pair_tag = lambda a, b: f"{NAME[a]} − {NAME[b]}"
cmp = pd.DataFrame(index=piv.index)
for a, b in PAIRS:
    cmp[f"d_mean ({pair_tag(a, b)})"] = piv["mean_error"][a] - piv["mean_error"][b]
    cmp[f"d_median ({pair_tag(a, b)})"] = piv["median"][a] - piv["median"][b]
    cmp[f"d_p90 ({pair_tag(a, b)})"] = piv["p90"][a] - piv["p90"][b]
    cmp[f"d_time_s ({NAME[b]} − {NAME[a]})"] = piv["total_s"][b] - piv["total_s"][a]

fig, ax = plt.subplots(figsize=(10, 4.8)); wbar = 0.27
for j, (a, b) in enumerate(PAIRS):
    ax.bar(cmp.index + (j - 1) * wbar, cmp[f"d_mean ({pair_tag(a, b)})"], width=wbar, color=PCOL[(a, b)], label=pair_tag(a, b))
ax.axhline(0, color="k", lw=.8)
ax.set_xticks(range(1, K_MAX + 1)); ax.set_xlabel("k"); ax.set_ylabel("Chênh lệch sai số trung bình (m)")
ax.set_title("Chênh lệch sai số trung bình giữa các cặp (dương = phương pháp thứ hai tốt hơn)"); ax.grid(alpha=.3, axis="y"); ax.legend()
fig.tight_layout(); fig.savefig("L4_C6_chenh_lech.png", dpi=150); plt.show()

# PHẦN D — Đánh giá: các phương pháp khác nhau thế nào?

### D1. So sánh ở cùng k (k = 1 → 20)
Bảng hiệu số sai số trung bình (mét) của ba cặp ở từng k, **không phụ thuộc** việc chọn k trên tập test. Giá trị **dương = phương pháp thứ hai trong cặp chính xác hơn**. k = 1 cả ba phương pháp giống hệt nhau nên chỉ xét k ≥ 2 khi đếm.

In [ ]:
d1 = pd.DataFrame({pair_tag(a, b): piv["mean_error"][a] - piv["mean_error"][b] for a, b in PAIRS})
display(d1.round(4))
n_ge2 = len(d1) - 1
for a, b in PAIRS:
    v = d1[pair_tag(a, b)].iloc[1:]
    win, lose = int((v > 1e-9).sum()), int((v < -1e-9).sum())
    print(f"{pair_tag(a, b)} (k = 2..{K_MAX}): {NAME[b]} tốt hơn ở {win}/{n_ge2} giá trị k, kém hơn ở {lose}/{n_ge2}; chênh lệch trung bình qua các k = {v.mean():+.3f} m")

### D2. So sánh từng mẫu + kiểm định thống kê (Wilcoxon cặp)
Với cùng một mẫu kiểm tra, xem phương pháp nào cho sai số nhỏ hơn, rồi dùng **Wilcoxon cặp** để biết chênh lệch có ý nghĩa thống kê hay chỉ là ngẫu nhiên. Làm tại k tốt nhất của WKNN (1/d) và của WKNN (1/d²) (nếu hai k trùng nhau thì chỉ làm một lần). Vì so 3 cặp nên ngưỡng hiệu chỉnh là **0,05 / 3**.

In [ ]:
from scipy.stats import wilcoxon
ALPHA = 0.05 / len(PAIRS)
K_LIST = sorted({BEST["idw"], BEST["idw2"]})
res = []
for K_CMP in K_LIST:
    oks = [R[(m, K_CMP)][0] for m in METHODS]
    assert all((o == oks[0]).all() for o in oks)           # vote toa/tang giong nhau -> cung tap mau dung
    ok = oks[0]
    print(f"\n=== Tại k = {K_CMP} ({ok.sum()} mẫu đúng tòa + tầng) ===")
    for a, b in PAIRS:
        ea, eb = R[(a, K_CMP)][1][ok], R[(b, K_CMP)][1][ok]; dl = ea - eb
        p_b, p_a, p_t = 100*(dl > 1e-9).mean(), 100*(dl < -1e-9).mean(), 100*(abs(dl) <= 1e-9).mean()
        try: pv = wilcoxon(ea, eb).pvalue
        except ValueError: pv = None
        kq = ("không tính được" if pv is None else ("CÓ ý nghĩa thống kê" if pv < ALPHA else "KHÔNG có ý nghĩa thống kê") + f" (ngưỡng {ALPHA:.4f})")
        print(f"{pair_tag(a, b)}: {NAME[b]} nhỏ hơn ở {p_b:.1f}% mẫu | lớn hơn {p_a:.1f}% | bằng {p_t:.1f}% ; "
              f"mean {ea.mean():.3f} vs {eb.mean():.3f} m (chênh {dl.mean():+.3f} m) ; Wilcoxon p = {('NA' if pv is None else f'{pv:.3g}')} → {kq}")
        res.append(dict(k=K_CMP, a=a, b=b, cap=pair_tag(a, b), mean_a=ea.mean(), mean_b=eb.mean(), diff_a_minus_b=dl.mean(),
                        pct_b_better=p_b, pct_a_better=p_a, pct_tie=p_t, p_value=pv, significant=(pv is not None and pv < ALPHA)))
d2 = pd.DataFrame(res)

### D3. Kết luận tự động
Tóm tắt ba tiêu chí: success rate, sai số vị trí (ở cùng k = k tốt nhất của WKNN (1/d²)), thời gian. Dòng cuối trả lời câu hỏi chính: **tăng bậc trọng số từ 1 lên 2 có làm khác đi không**.

In [ ]:
K0 = BEST["idw2"]
bst = {m: sweep[(sweep.method == m) & (sweep.k == BEST[m])].iloc[0] for m in METHODS}
at0 = {m: sweep[(sweep.method == m) & (sweep.k == K0)].iloc[0] for m in METHODS}
print("=" * 78)
print("1) SUCCESS RATE (đúng tòa + tầng)")
print("   " + " | ".join(f"{NAME[m]} (k={BEST[m]}): {bst[m].success_rate:.2f}%" for m in METHODS))
print("   → Bằng nhau ở cùng k vì vote tòa/tầng giống hệt nhau; kiểu trọng số không ảnh hưởng tiêu chí này.")
print("2) SAI SỐ VỊ TRÍ")
print("   Mỗi phương pháp ở k tốt nhất riêng: " + " | ".join(f"{NAME[m]} {bst[m].mean_error:.3f} m" for m in METHODS))
print(f"   Cùng k = {K0}: " + " | ".join(f"{NAME[m]} {at0[m].mean_error:.3f} m" for m in METHODS))
for a, b in PAIRS:
    r = d2[(d2.a == a) & (d2.b == b) & (d2.k == K0)].iloc[0]
    diff, rel = r.diff_a_minus_b, 100 * r.diff_a_minus_b / r.mean_a
    if r.significant: verdict = f"{NAME[b]} TỐT HƠN có ý nghĩa thống kê" if diff > 0 else f"{NAME[b]} KÉM HƠN có ý nghĩa thống kê"
    else: verdict = "KHÁC BIỆT KHÔNG có ý nghĩa thống kê (gần như tương đương)"
    print(f"   → {pair_tag(a, b)} = {diff:+.3f} m ({rel:+.1f}%; dương = {NAME[b]} chính xác hơn): {verdict}.")
print("3) THỜI GIAN")
print("   Tổng: " + " | ".join(f"{NAME[m]} {bst[m].total_s:.2f} s" for m in METHODS))
print("   Riêng bước gộp tọa độ: " + " | ".join(f"{NAME[m]} {bst[m].t_predict_s*1000:.1f} ms" for m in METHODS))
big = max(bst[m].total_s for m in METHODS) / min(bst[m].total_s for m in METHODS) - 1
print("   → Tổng thời gian chênh dưới 5%: các dạng trọng số không làm hệ thống chậm đáng kể (bước tìm láng giềng chiếm phần lớn và giống nhau)." if big < 0.05
      else f"   → Tổng thời gian chênh lệch lớn nhất giữa các phương pháp: {100*big:.1f}%.")
print("=" * 78)
print("Lưu ý: k chọn trên tập test nên kết quả hơi lạc quan; kết luận chính nên dựa trên so sánh cùng k (D1, D2).")

# PHẦN E — Lưu kết quả
Lưu các bảng ra file CSV (để đưa vào báo cáo) và tải về cùng các biểu đồ PNG đã vẽ ở Phần C.

In [ ]:
sweep.to_csv("L4_khao_sat_k_knn_wknn.csv", index=False)
table.to_csv("L4_bang_tong_hop.csv", index=False)
d1.to_csv("L4_so_sanh_cung_k.csv")
d2.to_csv("L4_kiem_dinh_wilcoxon.csv", index=False)
files_out = ["L4_khao_sat_k_knn_wknn.csv", "L4_bang_tong_hop.csv", "L4_so_sanh_cung_k.csv", "L4_kiem_dinh_wilcoxon.csv",
             "L4_C1_success_rate.png", "L4_C2_mean_error.png", "L4_C3_median.png", "L4_C4_cdf.png", "L4_C5_time.png", "L4_C6_chenh_lech.png"]
print("Đã lưu:", ", ".join(files_out))
try:
    from google.colab import files
    for f in files_out: files.download(f)
except Exception: pass